# Векторизация текстов и сравнение их близости

Четыре текста на русском языке:
- `text1_machine_learning.txt` — Машинное обучение
- `text2_neural_networks.txt` — Нейронные сети и глубокое обучение
- `text3_russian_literature.txt` — Золотой век русской литературы
- `text4_russian_cuisine.txt` — Русская кухня

Метод векторизации: **TF-IDF** (Term Frequency – Inverse Document Frequency).
Мера близости: **косинусное сходство** (cosine similarity).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
files = [
    ('text1_machine_learning.txt',  'ML'),
    ('text2_neural_networks.txt',   'Нейросети'),
    ('text3_russian_literature.txt','Литература'),
    ('text4_russian_cuisine.txt',   'Кухня'),
]

texts = []
labels = []

for fname, label in files:
    with open(fname, encoding='utf-8') as f:
        texts.append(f.read())
    labels.append(label)

for label, text in zip(labels, texts):
    print(f'{label}: {len(text)} символов')

## TF-IDF векторизация

TF-IDF взвешивает каждое слово по двум факторам:
- **TF** — как часто слово встречается в данном документе
- **IDF** — насколько редко слово встречается во всей коллекции (редкие слова несут больше информации)

В итоге каждый текст представляется вектором в пространстве всех уникальных слов коллекции.

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer='word',
    token_pattern=r'[а-яёА-ЯЁa-zA-Z]{2,}',  # только слова длиной ≥2, без цифр и пунктуации
    min_df=1,
    sublinear_tf=True  # log(1 + tf) вместо tf — сглаживает влияние очень частых слов
)

tfidf_matrix = vectorizer.fit_transform(texts)
print(f'Размер матрицы TF-IDF: {tfidf_matrix.shape}')  # (4 текста, N уникальных слов)

## Матрица косинусного сходства

In [ ]:
sim_matrix = cosine_similarity(tfidf_matrix)

print('Матрица косинусного сходства:')
header = f'{"":12}' + ''.join(f'{l:12}' for l in labels)
print(header)
for i, row_label in enumerate(labels):
    row = f'{row_label:12}' + ''.join(f'{sim_matrix[i, j]:.4f}      ' for j in range(len(labels)))
    print(row)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    sim_matrix,
    annot=True,
    fmt='.4f',
    xticklabels=labels,
    yticklabels=labels,
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    ax=ax
)
ax.set_title('Косинусное сходство между текстами (TF-IDF)', pad=14)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150)
plt.show()
print('Тепловая карта сохранена в similarity_heatmap.png')

In [ ]:
# Вывести все пары в порядке убывания сходства (исключая диагональ)
n = len(labels)
pairs = []
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((labels[i], labels[j], sim_matrix[i, j]))

pairs.sort(key=lambda x: x[2], reverse=True)

print('Пары текстов по убыванию косинусного сходства:')
for a, b, score in pairs:
    print(f'  {a} — {b}: {score:.4f}')

## Анализ результатов

### Ожидали ли мы такого результата?

**Да, в целом результат полностью ожидаем.**

Наиболее близкими оказались тексты **«Машинное обучение»** и **«Нейронные сети»** — и это логично. Оба текста принадлежат одной предметной области (искусственный интеллект, обучение по данным), а TF-IDF чувствителен именно к словарному составу. В обоих текстах часто встречаются слова: *алгоритм*, *данные*, *модель*, *обучение*, *сеть*, *слой*, *параметр*, *функция*, *задача*, *метод*, *классификация*. Именно эти слова создали высокое косинусное сходство.

Тексты **«Литература»** и **«Кухня»** схожи друг с другом чуть сильнее, чем с текстами об ИИ, потому что у них есть общий пласт гуманитарной лексики: упоминание исторических периодов, слова *русский*, *традиция*, *период*, *история* и т.п. Однако их доминирующая лексика принципиально разная (Пушкин, роман, герой vs. хлеб, суп, блюдо, вкус), поэтому их сходство между собой тоже невелико.

### Почему TF-IDF даёт именно такую картину?

TF-IDF — **пространство слов без учёта семантики**: он не знает, что *нейрон* и *синапс* тематически близки к *алгоритму* и *сети*, если эти слова не встречаются в обоих текстах. Поэтому для тематически родственных, но не дублирующих текстов он может недооценивать сходство. При этом метод отлично работает именно тогда, когда авторы по одной теме используют схожий специализированный словарь — что и происходит с нашими текстами об ИИ.

Если бы мы использовали **эмбеддинги** (word2vec, fastText, sentence-BERT), семантически близкие тексты получали бы бо́льшее сходство даже при разном конкретном словарном составе. Но для данной задачи TF-IDF достаточно наглядно показывает интуитивно ожидаемую картину.